# Random Forest — multibranch-style sequence classifier

This notebook uses the cleaned `base_utils_qwen.py` and trains a sequence-level Random Forest.

Local quick runs use `data/sample.csv` (37 sequences). Set `use_sample_data = False` for full `train.csv`.

Switch `search_mode` between `'grid'` and `'bayesian'` in the config cell.

Style:
- configuration
- data loading
- split
- estimator
- parameter search (grid or Bayesian)
- holdout evaluation
- save results


In [1]:
import os
import sys
import warnings
import importlib
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# Local path routing (works from notebooks/ or project root)
current_dir = os.getcwd()
workspace_root = current_dir
if os.path.basename(current_dir) == 'notebooks':
    workspace_root = os.path.dirname(current_dir)

src_path = os.path.join(workspace_root, 'src')
sys.path.insert(0, workspace_root)
sys.path.insert(0, src_path)

# Kaggle path routing
try:
    dataset_name = os.listdir('/kaggle/input/datasets/keithmarange')[0]
    sys.path.append(f'/kaggle/input/datasets/keithmarange/{dataset_name}/')
    sys.path.append('/kaggle/input/cmi-competition-code')
except Exception:
    pass

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, GroupKFold, GroupShuffleSplit
from sklearn.metrics import f1_score, make_scorer
from sklearn.pipeline import Pipeline

try:
    import skopt
    from skopt import BayesSearchCV
    from skopt.space import Categorical, Integer, Real
    SKOPT_AVAILABLE = True
except ImportError:
    BayesSearchCV = None
    Categorical = Integer = Real = None
    SKOPT_AVAILABLE = False

try:
    import src.base_utils_qwen as base_utils_qwen
    importlib.reload(base_utils_qwen)
    from src import data_utils
    from src.base_utils_qwen import (
        SequenceExtractor,
        RandomForestSequenceClassifier,
        competition_scorer,
        evaluate_holdout,
        make_competition_scorer,
        prepare_bayesian_space,
        SensorAugmentor
    )
    print('Imports loaded from src/ (reloaded)')
except ImportError:
    import data_utils
    import importlib
    importlib.reload(sys.modules.get('base_utils_qwen', __import__('base_utils_qwen')))
    from base_utils_qwen import (
        SequenceExtractor,
        RandomForestSequenceClassifier,
        competition_scorer,
        evaluate_holdout,
        make_competition_scorer,
        prepare_bayesian_space,
        SensorAugmentor
    )
    print('Imports loaded from flat src path (reloaded)')


Imports loaded from flat src path (reloaded)


In [2]:
# Install optional search / feature dependencies if missing (safe to re-run)
for package_name, import_name in [
    ('scikit-optimize', 'skopt'),
    ('PyWavelets', 'pywt'),
]:
    try:
        __import__(import_name)
    except ImportError:
        import subprocess
        import sys
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package_name])
        print(f'Installed {package_name}')

try:
    import skopt
    from skopt import BayesSearchCV
    from skopt.space import Categorical, Integer, Real
    SKOPT_AVAILABLE = True
    print(f'scikit-optimize {skopt.__version__} ready')
except ImportError:
    BayesSearchCV = None
    Categorical = Integer = Real = None
    SKOPT_AVAILABLE = False
    print('scikit-optimize unavailable')

try:
    import pywt
    print(f'PyWavelets {pywt.__version__} ready')
except ImportError:
    print('PyWavelets unavailable')

scikit-optimize 0.10.2 ready
PyWavelets 1.8.0 ready


In [3]:
experiment_name = 'random_forest'

TARGET_COL = 'bfrb'

# Use sample.csv for a quick smoke test; set False for full train.csv
use_sample_data = True
sample_file = 'sample.csv'  # also available: eg.csv

search_mode = 'grid'  # 'grid' or 'bayesian'; Bayesian smoke test below uses sample data
random_state = 42
n_splits = 1          # 1 -> GroupShuffleSplit; >=2 -> GroupKFold
cv_test_size = 0.4
train_size = 0.6
n_iter = 13            # Bayesian iterations for a quick smoke test
verbose = 3
error_score = 'raise'  # 'raise' or 'warn'

orientation_filter_list = [] # ['Seated Lean Non Dom - FACE DOWN', 'Lie on Side - Non Dominant', 'Seated Straight', 'Lie on Back' ]

problematic_sequence_bool = False
target_only_bool = False

results_dir = Path(f'results_{experiment_name}')
results_dir.mkdir(exist_ok=True)
timestamp = datetime.now().strftime('%Y%m%d_%H%M')

if TARGET_COL == 'bfrb':
    scorer = competition_scorer
else:
    scorer = make_scorer(f1_score, average='macro', zero_division=0)

search_mode = str(search_mode).lower()
if search_mode in ('bayes', 'bayesian'):
    search_mode = 'bayesian'
elif search_mode != 'grid':
    raise ValueError("search_mode must be 'grid' or 'bayesian'")

if n_splits <= 1:
    cv_object = GroupShuffleSplit(
        n_splits=1,
        test_size=cv_test_size,
        random_state=random_state,
    )
else:
    cv_object = GroupKFold(n_splits=n_splits)


In [4]:
data_root = data_utils.find_data_root()
sample_path = data_root / sample_file

if use_sample_data and sample_path.exists():
    raw_train_df = pd.read_csv(sample_path)
    print(f'Using {sample_file}: {raw_train_df["sequence_id"].nunique()} sequences')
else:
    raw_train_df = pd.read_csv(data_root / 'train.csv')
    print(f'Using train.csv: {raw_train_df["sequence_id"].nunique()} sequences')

train_demo_df = pd.read_csv(data_root / 'train_demographics.csv')

train_df = raw_train_df.set_index('row_id').copy(deep=True)

train_df['gesture'] = train_df['gesture'].fillna('non_bfrb').astype(str)
train_df['orientation'] = train_df['orientation'].fillna('Unknown').astype(str)
train_df['is_target'] = train_df['sequence_type'].eq('Target').astype(int)
train_df['bfrb'] = train_df['gesture'].where(train_df['is_target'].astype(bool), 'non_bfrb')

train_df['gesture_position'] = train_df['gesture'].str.split(' - ').str[0]
train_df['gesture_action'] = train_df['gesture'].str.split(' - ').str[-1]

problematic_sequence_df = train_df.groupby('sequence_id')[['acc_x', 'acc_y', 'acc_z', 'rot_x', 'rot_y', 'rot_w', 'rot_z']].skew().abs()
ideal_skew_threshold = 1.6
problematic_features_threshold = 3
result_series = ((problematic_sequence_df > ideal_skew_threshold).sum(axis=1) >= problematic_features_threshold)
problematic_sequences_list = result_series.loc[result_series].index
train_df['problematic_sequence'] = train_df['sequence_id'].isin(problematic_sequences_list).astype(bool)

if orientation_filter_list:
    train_df = train_df[train_df['orientation'].isin(orientation_filter_list)].copy()

if problematic_sequence_bool:
    train_df = train_df[train_df['problematic_sequence']].copy()

if target_only_bool:
    train_df = train_df[train_df['is_target']].copy()

train_df[TARGET_COL] = train_df[TARGET_COL].fillna('non_bfrb').astype(str)


Using Kaggle data folder: /kaggle/input/competitions/cmi-detect-behavior-with-sensor-data
Using train.csv: 8151 sequences


In [5]:
try:
    train_sample_df, hold_out_df = data_utils.sample_balanced_split(
        train_df,
        train_pct=train_size,
        test_pct=min(0.2, 1 - train_size),
        random_state=random_state,
    )
except Exception:
    seq_df = train_df[['sequence_id', 'is_target', TARGET_COL]].drop_duplicates('sequence_id').sort_values('sequence_id')
    gss = GroupShuffleSplit(n_splits=1, train_size=train_size, random_state=random_state)
    train_idx, test_idx = next(gss.split(seq_df, groups=seq_df['sequence_id']))

    train_seqs = seq_df.iloc[train_idx]['sequence_id']
    test_seqs = seq_df.iloc[test_idx]['sequence_id']

    train_sample_df = train_df[train_df['sequence_id'].isin(train_seqs)].copy()
    hold_out_df = train_df[train_df['sequence_id'].isin(test_seqs)].copy()

X_train = train_sample_df.copy()
X_test = hold_out_df.copy()

y_train = X_train[['sequence_id', 'is_target', TARGET_COL]].copy()
y_test = X_test[['sequence_id', 'is_target', TARGET_COL]].copy()

groups = X_train['sequence_id'].astype(str)

print('Train sequences:', X_train['sequence_id'].nunique())
print('Test sequences:', X_test['sequence_id'].nunique())


Train: 3879 seqs | 47.6%
Test:  966 seqs  | 11.9%
Train sequences: 3879
Test sequences: 966


In [6]:
rf_pipeline = Pipeline([
    ('augmentor', SensorAugmentor(
        sequence_col='sequence_id',
        counter_col='sequence_counter',
        prob=0.0,
        per_aug_prob=0.0,
    )),
    ('estimator', RandomForestSequenceClassifier(
        primary_target=TARGET_COL,
        extractor=SequenceExtractor(
            output_format='frame',
            acc_modes='raw|velocity|jerk',
            rotation_modes='quaternion|angular_velocity',
        ),
        estimator=RandomForestClassifier(
            n_estimators=300,
            random_state=random_state,
        ),
        random_state=random_state,
    )),
])

In [7]:
# ============================================================
# GRID SEARCH SPACE
# Practical compact grid. Bayesian space does the full exploration.
# ============================================================

GRID_PARAM_SPACE = {
    # ------------------------------------------------------------
    # EXTRACTOR: main sensor-feature domains
    # ------------------------------------------------------------
    'estimator__extractor__acc_modes': [
        'smoothed|velocity|displacement|jerk',
        None
    ],
    'estimator__extractor__rotation_modes': [
        'quaternion|euler|angular_velocity',
        None
    ],
    'estimator__extractor__tof_modes': [
        'pooled_stats|sensor_stats',
        None
    ],
    'estimator__extractor__thm_modes': [
        'centered_diff',
    ],
    'estimator__extractor__frame_stats': [
        'mean,std,min,max,last,first,rms',
        None
    ],

    # ------------------------------------------------------------
    # EXTRACTOR: fixed preprocessing for stable RF smoke/grid runs
    # ------------------------------------------------------------
    'estimator__extractor__motion_filter_mode': [None],
    'estimator__extractor__use_dead_reckoning': [False],
    'estimator__extractor__dead_reckoning_detrend': [False],
    'estimator__extractor__kalman_process_noise': [1e-3],
    'estimator__extractor__kalman_measurement_noise': [1e-1],

    'estimator__extractor__window_size': [20],
    'estimator__extractor__smooth_alpha': [None],
    'estimator__extractor__clip_value': [None],
    'estimator__extractor__interp_mode': ['linear'],

    # ------------------------------------------------------------
    # EXTRACTOR: fixed frame-output safety params
    # ------------------------------------------------------------
    'estimator__extractor__output_format': ['frame'],
    'estimator__extractor__padding_value': [0.0],
    'estimator__extractor__maxlen': [160],
    'estimator__extractor__chunk_window_size': [50],
    'estimator__extractor__chunk_stride': [25],
    'estimator__extractor__add_global_context': [False],
    'estimator__extractor__resample_modalities': [False],
    'estimator__extractor__compute_dt': [True],

    'estimator__extractor__imu_native_sampling_rate': [100],
    'estimator__extractor__rot_native_sampling_rate': [100],
    'estimator__extractor__tof_native_sampling_rate': [20],
    'estimator__extractor__thm_native_sampling_rate': [20],

    'estimator__extractor__imu_target_sampling_rate': [100],
    'estimator__extractor__rot_target_sampling_rate': [100],
    'estimator__extractor__tof_target_sampling_rate': [20],
    'estimator__extractor__thm_target_sampling_rate': [20],

    # ------------------------------------------------------------
    # EXTRACTOR: STFT & CWT (Time-Frequency Domains) - FIXED FOR GRID
    # ------------------------------------------------------------
    'estimator__extractor__stft_nperseg': [100],
    'estimator__extractor__stft_noverlap': [50],
    'estimator__extractor__stft_window_type': ['hann'],
    'estimator__extractor__stft_use_log_scale': [True],

    'estimator__extractor__cwt_wavelet': ['morl'],
    'estimator__extractor__cwt_max_scale': [32],
    'estimator__extractor__cwt_n_scales': [32],
    'estimator__extractor__cwt_use_log_scale': [False],

    # ------------------------------------------------------------
    # AUGMENTOR: Baseline (Set to 0.0 to switch off)
    # Uses EXACT valid parameter names from your SensorAugmentor class
    # ------------------------------------------------------------
    'augmentor__prob': [0.0],
    'augmentor__per_aug_prob': [0.0],
    'augmentor__jitter_sigma': [0.0],
    'augmentor__noise_std': [0.0],
    'augmentor__scaling_sigma': [0.0],
    'augmentor__sensor_drop_prob': [0.0],
    'augmentor__channel_drop_prob': [0.0],
    'augmentor__time_shift_frac': [0.0],
    'augmentor__crop_frac_range': [(0.5, 1.0)],
    'augmentor__temporal_mask_frac': [0.0],
    'augmentor__temporal_num_masks': [0],
    'augmentor__warp_sigma': [0.0],
    'augmentor__warp_num_knots': [4],

    # ------------------------------------------------------------
    # RANDOM FOREST ESTIMATOR
    # ------------------------------------------------------------
    'estimator__estimator__n_estimators': [100],
    'estimator__estimator__criterion': ['gini'],
    'estimator__estimator__max_depth': [30],
    'estimator__estimator__min_samples_split': [15],
    'estimator__estimator__min_samples_leaf': [10],
    'estimator__estimator__max_features': ['sqrt'],
    'estimator__estimator__bootstrap': [True],
    'estimator__estimator__class_weight': ['balanced'],
}


# ============================================================
# BAYESIAN SEARCH SPACE
# Full exploration space.
# ============================================================

if SKOPT_AVAILABLE:
    try:
        BAYESIAN_PARAM_SPACE = {
            # --------------------------------------------------------
            # EXTRACTOR: sensor-feature domains
            # --------------------------------------------------------
            'estimator__extractor__acc_modes': Categorical([
                'raw',
                'raw|velocity',
                'smoothed|velocity|displacement|jerk',
                None
            ]),
            'estimator__extractor__rotation_modes': Categorical([
                'quaternion|euler|angular_velocity',
                'quaternion|angular_velocity|delta_euler|rot6d',
                None
            ]),
            'estimator__extractor__tof_modes': Categorical([
                'pooled_stats|sensor_stats',
                None
            ]),
            'estimator__extractor__thm_modes': Categorical([
                'centered_diff',
            ]),
            'estimator__extractor__frame_stats': Categorical([
                'mean,std,min,max,last',
                'mean,std,min,max,last,first,rms,abs_mean',
                None
            ]),

            # --------------------------------------------------------
            # EXTRACTOR: preprocessing / filtering
            # --------------------------------------------------------
            'estimator__extractor__motion_filter_mode': Categorical([
                'extended_kalman',
            ]),
            'estimator__extractor__use_dead_reckoning': Categorical([True]),
            'estimator__extractor__dead_reckoning_detrend': Categorical([True]),

            'estimator__extractor__kalman_process_noise': Real(1e-5, 1e-1, prior='log-uniform'),
            'estimator__extractor__kalman_measurement_noise': Real(1e-3, 1e1, prior='log-uniform'),

            'estimator__extractor__window_size': Categorical([5]),
            'estimator__extractor__smooth_alpha': Categorical([None, 0.20, 0.90]),
            'estimator__extractor__clip_value': Categorical([150.0]),
            'estimator__extractor__interp_mode': Categorical(['linear']),

            # --------------------------------------------------------
            # EXTRACTOR: STFT & CWT (Time-Frequency Domains) - BAYESIAN
            # --------------------------------------------------------
            'estimator__extractor__stft_nperseg': Categorical([32, 64, 128]),
            'estimator__extractor__stft_noverlap': Categorical([8, 16, 32]),
            'estimator__extractor__stft_window_type': Categorical(['hann', 'hamming', 'blackman']),
            'estimator__extractor__stft_use_log_scale': Categorical([True, False]),

            'estimator__extractor__cwt_wavelet': Categorical(['morl', 'mexh', 'gaus1', 'gaus2']),
            'estimator__extractor__cwt_max_scale': Categorical([64, 128, 256]),
            'estimator__extractor__cwt_n_scales': Categorical([16, 32, 64]),
            'estimator__extractor__cwt_use_log_scale': Categorical([True, False]),

            # --------------------------------------------------------
            # EXTRACTOR: fixed frame-output safety params
            # --------------------------------------------------------
            'estimator__extractor__output_format': Categorical(['frame']),
            'estimator__extractor__padding_value': Categorical([0.0]),
            'estimator__extractor__maxlen': Categorical([150]),
            'estimator__extractor__chunk_window_size': Categorical([100]),
            'estimator__extractor__chunk_stride': Categorical([50]),
            'estimator__extractor__add_global_context': Categorical([True]),
            'estimator__extractor__compute_dt': Categorical([True]),

            'estimator__extractor__imu_native_sampling_rate': Categorical([100]),
            'estimator__extractor__rot_native_sampling_rate': Categorical([100]),
            'estimator__extractor__tof_native_sampling_rate': Categorical([20]),
            'estimator__extractor__thm_native_sampling_rate': Categorical([20]),

            'estimator__extractor__imu_target_sampling_rate': Categorical([100]),
            'estimator__extractor__rot_target_sampling_rate': Categorical([100]),
            'estimator__extractor__tof_target_sampling_rate': Categorical([20]),
            'estimator__extractor__thm_target_sampling_rate': Categorical([20]),
            'estimator__extractor__resample_modalities': Categorical([True]),

            # --------------------------------------------------------
            # AUGMENTOR: Full Exploration
            # Uses EXACT valid parameter names from your SensorAugmentor class
            # --------------------------------------------------------
            'augmentor__prob': Real(0.0, 0.6),
            'augmentor__per_aug_prob': Real(0.0, 0.6),
            'augmentor__jitter_sigma': Real(0.0, 0.1),
            'augmentor__noise_std': Real(0.0, 0.1),
            'augmentor__scaling_sigma': Real(0.0, 0.2),
            'augmentor__sensor_drop_prob': Real(0.0, 0.3),
            'augmentor__channel_drop_prob': Real(0.0, 0.3),
            'augmentor__time_shift_frac': Real(0.0, 0.2),
            'augmentor__crop_frac_range': Categorical(['(0.5, 1.0)', '(0.7, 1.0)', '(0.8, 1.0)']),
            'augmentor__temporal_mask_frac': Real(0.0, 0.25),
            'augmentor__temporal_num_masks': Integer(1, 3),
            'augmentor__warp_sigma': Real(0.0, 0.3),
            'augmentor__warp_num_knots': Integer(2, 6),

            # --------------------------------------------------------
            # RANDOM FOREST ESTIMATOR
            # --------------------------------------------------------
            'estimator__estimator__n_estimators': Integer(5, 200),
            'estimator__estimator__criterion': Categorical(['gini', 'entropy']),
            'estimator__estimator__max_depth': Categorical([None, 10, 30, 50, 100]),
            'estimator__estimator__min_samples_split': Integer(2, 20),
            'estimator__estimator__min_samples_leaf': Integer(1, 8),
            'estimator__estimator__max_features': Categorical([0.3, 0.5, 0.7]),
            'estimator__estimator__bootstrap': Categorical([True]),
            'estimator__estimator__class_weight': Categorical(['balanced']),
            'estimator__estimator__min_impurity_decrease': Real(0.0, 0.005),
        }

    except Exception:
        BAYESIAN_PARAM_SPACE = GRID_PARAM_SPACE
else:
    BAYESIAN_PARAM_SPACE = GRID_PARAM_SPACE

BAYESIAN_PARAM_SPACE = prepare_bayesian_space(BAYESIAN_PARAM_SPACE)
param_space = BAYESIAN_PARAM_SPACE if search_mode == 'bayesian' else GRID_PARAM_SPACE


In [8]:
if search_mode == 'bayesian':
    if not SKOPT_AVAILABLE:
        raise ImportError(
            "Bayesian search requires scikit-optimize. "
            "Install with: pip install scikit-optimize"
        )

    search = BayesSearchCV(
        estimator=rf_pipeline,
        search_spaces=param_space,
        n_iter=n_iter,
        scoring=scorer,
        cv=cv_object,
        n_jobs=1,
        random_state=random_state,
        verbose=verbose,
        return_train_score=True,
        error_score=error_score,
    )
else:
    search = GridSearchCV(
        estimator=rf_pipeline,
        param_grid=param_space,
        scoring=scorer,
        cv=cv_object,
        n_jobs=1,
        verbose=verbose,
        return_train_score=True,
        error_score=error_score,
    )

search.fit(X_train, y_train, groups=groups)

print('Best CV score:', search.best_score_)
print('Best params:', search.best_params_)


Fitting 1 folds for each of 16 candidates, totalling 16 fits
[CV 1/1] END augmentor__channel_drop_prob=0.0, augmentor__crop_frac_range=(0.5, 1.0), augmentor__jitter_sigma=0.0, augmentor__noise_std=0.0, augmentor__per_aug_prob=0.0, augmentor__prob=0.0, augmentor__scaling_sigma=0.0, augmentor__sensor_drop_prob=0.0, augmentor__temporal_mask_frac=0.0, augmentor__temporal_num_masks=0, augmentor__time_shift_frac=0.0, augmentor__warp_num_knots=4, augmentor__warp_sigma=0.0, estimator__estimator__bootstrap=True, estimator__estimator__class_weight=balanced, estimator__estimator__criterion=gini, estimator__estimator__max_depth=30, estimator__estimator__max_features=sqrt, estimator__estimator__min_samples_leaf=10, estimator__estimator__min_samples_split=15, estimator__estimator__n_estimators=100, estimator__extractor__acc_modes=smoothed|velocity|displacement|jerk, estimator__extractor__add_global_context=False, estimator__extractor__chunk_stride=25, estimator__extractor__chunk_window_size=50, esti

In [9]:
best_model = search.best_estimator_

y_pred = best_model.predict(X_test)

eval_results = evaluate_holdout(
    y_test,
    y_pred,
    target_col=TARGET_COL,
    verbose=True,
)

print('Holdout competition score:', eval_results['competition_score'])

cv_df = pd.DataFrame(search.cv_results_)
cv_df.to_csv(results_dir / f'rf_multibranch_style_cv_{timestamp}.csv', index=False)

eval_results['results_df'].to_csv(
    results_dir / f'rf_multibranch_style_holdout_{timestamp}.csv',
    index=False,
)

pd.DataFrame(
    [
        {
            'best_score': search.best_score_,
            'best_params': str(search.best_params_),
            'holdout_score': eval_results['competition_score'],
        }
    ]
).to_csv(
    results_dir / f'rf_multibranch_style_best_{timestamp}.csv',
    index=False,
)



FINAL EVALUATION
Binary F1 (non_bfrb vs bfrb): 0.9374
BFRB Gesture Macro F1: 0.3968
COMPETITION SCORE: 0.6671

----------------------------------------
BFRB Gesture Classification Report
----------------------------------------
                          precision    recall  f1-score   support

   Above ear - pull hair       0.56      0.58      0.57        81
      Cheek - pinch skin       0.47      0.40      0.43        81
     Eyebrow - pull hair       0.36      0.15      0.21        81
     Eyelash - pull hair       0.46      0.35      0.39        81
Forehead - pull hairline       0.47      0.43      0.45        81
      Forehead - scratch       0.53      0.72      0.61        81
       Neck - pinch skin       0.35      0.37      0.36        81
          Neck - scratch       0.54      0.56      0.55        81
                non_bfrb       0.00      0.00      0.00         0

                accuracy                           0.44       648
               macro avg       0.42      0.

In [10]:
best_estimator = best_model.named_steps['estimator']
importances = pd.Series(
    best_estimator.estimator_.feature_importances_,
    index=best_estimator.extractor_.frame_feature_names_,
).sort_values(ascending=False)

print(importances.head(50))

tof_3_mean_min                      0.015814
tof_3_mean_std                      0.010098
tof_3_min_last                      0.008569
tof_2_mean_mean                     0.007665
tof_2_mean_min                      0.007616
thm_2_centered_min                  0.007026
thm_2_centered_std                  0.006949
tof_3_mean_rms                      0.006865
tof_4_mean_mean                     0.006707
tof_3_mean_mean                     0.006651
tof_2_mean_std                      0.006520
tof_3_mean_last                     0.006410
tof_3_std_max                       0.006241
tof_2_mean_rms                      0.006235
thm_2_centered_rms                  0.005786
tof_4_std_rms                       0.005669
tof_4_std_mean                      0.005614
tof_3_std_rms                       0.005421
tof_4_mean_rms                      0.005379
thm_2_centered_diff_std             0.005351
tof_1_mean_min                      0.005323
thm_2_centered_diff_rms             0.005286
tof_3_min_